# Markdown 英译中 GPU 翻译器

使用 Helsinki-NLP/opus-mt-en-zh 模型，在 Colab GPU 上批量翻译 Markdown 文件。

**使用步骤：**
1. 运行「环境检查」单元格，确认 GPU 可用
2. 运行「安装依赖」单元格
3. 运行「上传文件」单元格，选择本地 .md 文件上传
4. 运行「加载模型」单元格
5. 运行「执行翻译」单元格
6. 运行「下载结果」单元格，获取翻译后的中文 Markdown 文件

In [2]:
#@title 1. 环境检查
import torch

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    #print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')
else:
    print('WARNING: GPU not available! Go to Runtime > Change runtime type > T4 GPU')

GPU: Tesla T4


In [3]:
#@title 2. 安装依赖
!pip install transformers sentencepiece sacremoses -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 51.3 MB/s eta 0:00:00


In [5]:
#@title 3. 上传文件
from google.colab import files
import os

uploaded = files.upload()
INPUT_FILE = list(uploaded.keys())[0]
print(f'Uploaded: {INPUT_FILE} ({os.path.getsize(INPUT_FILE) / 1024:.1f} KB)')

Saving input (1).md to input (1).md
Uploaded: input (1).md (661.2 KB)


In [6]:
#@title 4. 加载模型
import torch
from transformers import MarianMTModel, MarianTokenizer

MODEL_NAME = 'Helsinki-NLP/opus-mt-en-zh'
BATCH_SIZE = 32
MAX_LENGTH = 512

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

print(f'Loading model {MODEL_NAME}...')
tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
model = MarianMTModel.from_pretrained(MODEL_NAME).to(device)
model.eval()
print('Model loaded.')

Using device: cuda
Loading model Helsinki-NLP/opus-mt-en-zh...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/806k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/805k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Model loaded.


In [7]:
#@title 5. 定义翻译函数
import re
import sys
import time

def protect_special(text):
    protected = {}
    counter = [0]

    def _save(m):
        key = f'XPROT{counter[0]}X'
        protected[key] = m.group(0)
        counter[0] += 1
        return key

    text = re.sub(r'```[\s\S]*?```', _save, text)
    text = re.sub(r'`[^`]+`', _save, text)
    text = re.sub(r'\!\[.*?\]\(.*?\)', _save, text)
    text = re.sub(r'\$\$[\s\S]*?\$\$', _save, text)
    text = re.sub(r'(?<!\$)\$[^$\n]+?\$(?!\$)', _save, text)
    text = re.sub(r'\\\[.*?\\\]', _save, text)
    text = re.sub(r'\\\([^)]*?\\\)', _save, text)
    text = re.sub(r'\\boxed\{[^}]*\}', _save, text)
    text = re.sub(r'https?://\S+', _save, text)

    return text, protected


def restore_special(text, protected):
    for key, val in sorted(protected.items(), key=lambda x: -len(x[0])):
        text = text.replace(key, val)
    return text


def parse_blocks(lines):
    blocks = []
    i = 0
    while i < len(lines):
        line = lines[i]
        stripped = line.strip()

        if stripped.startswith('```'):
            code_lines = [line]
            i += 1
            while i < len(lines) and not lines[i].strip().startswith('```'):
                code_lines.append(lines[i])
                i += 1
            if i < len(lines):
                code_lines.append(lines[i])
                i += 1
            blocks.append(('code', ''.join(code_lines)))
        elif not stripped:
            blocks.append(('blank', line))
            i += 1
        elif re.match(r'^!\[.*?\]\(.*?\)', stripped):
            blocks.append(('image', line))
            i += 1
        elif re.match(r'^\|[\s\-:|]+\|$', stripped):
            blocks.append(('table_sep', line))
            i += 1
        else:
            text_lines = []
            while i < len(lines):
                s = lines[i].strip()
                if s.startswith('```') or not s:
                    break
                if re.match(r'^!\[.*?\]\(.*?\)', s):
                    break
                if re.match(r'^\|[\s\-:|]+\|$', s):
                    break
                text_lines.append(lines[i])
                i += 1
            blocks.append(('text', ''.join(text_lines)))

    return blocks


def extract_segments_from_block(text):
    segments = []
    lines = text.split('\n')

    for line in lines:
        if not line.strip():
            segments.append({'type': 'blank', 'line': line})
            continue

        stripped = line.strip()
        indent = len(line) - len(line.lstrip())

        prefix = ''
        content = stripped

        if stripped.startswith('#'):
            m = re.match(r'^(#+\s*)(.*)', stripped)
            if m:
                prefix = m.group(1)
                content = m.group(2)
        elif re.match(r'^[-*]\s+', stripped):
            m = re.match(r'^([-*]\s+)(.*)', stripped)
            if m:
                prefix = m.group(1)
                content = m.group(2)
        elif re.match(r'^\d+\.\s+', stripped):
            m = re.match(r'^(\d+\.\s+)(.*)', stripped)
            if m:
                prefix = m.group(1)
                content = m.group(2)
        elif stripped.startswith('|') and stripped.endswith('|'):
            cells = stripped.split('|')
            cell_infos = []
            for cell in cells:
                if cell.strip():
                    prot_content, prot = protect_special(cell)
                    cell_infos.append({
                        'type': 'cell',
                        'original': cell,
                        'content': prot_content,
                        'protected': prot,
                    })
                else:
                    cell_infos.append({'type': 'blank_cell', 'original': cell})
            segments.append({'type': 'table_row', 'cells': cell_infos, 'indent': indent})
            continue

        prot_content, prot = protect_special(content)
        segments.append({
            'type': 'text',
            'indent': indent,
            'prefix': prefix,
            'content': prot_content,
            'protected': prot,
        })

    return segments


def collect_translatable_texts(all_segments):
    texts = []
    for seg in all_segments:
        if seg['type'] == 'text':
            c = seg['content']
            if c and c.strip() and len(c.strip()) >= 2:
                texts.append(c)
            else:
                texts.append(None)
        elif seg['type'] == 'table_row':
            for cell in seg['cells']:
                if cell['type'] == 'cell':
                    c = cell['content']
                    if c and c.strip() and len(c.strip()) >= 2:
                        texts.append(c)
                    else:
                        texts.append(None)
    return texts


def batch_translate_gpu(texts):
    valid_indices = [i for i, t in enumerate(texts) if t is not None]
    valid_texts = [texts[i] for i in valid_indices]

    if not valid_texts:
        return [None] * len(texts)

    results = [None] * len(texts)
    total = len(valid_texts)
    done = 0

    with torch.no_grad():
        for start in range(0, total, BATCH_SIZE):
            batch = valid_texts[start:start + BATCH_SIZE]

            try:
                inputs = tokenizer(
                    batch,
                    return_tensors='pt',
                    padding=True,
                    truncation=True,
                    max_length=MAX_LENGTH,
                ).to(device)

                outputs = model.generate(
                    **inputs,
                    max_length=MAX_LENGTH,
                    num_beams=4,
                    no_repeat_ngram_size=3,
                )

                decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

                for j, translated in enumerate(decoded):
                    results[valid_indices[start + j]] = translated

            except Exception as e:
                print(f'Batch error at {start}: {e}', file=sys.stderr)
                for j in range(len(batch)):
                    results[valid_indices[start + j]] = batch[j]

            done += len(batch)
            if done % (BATCH_SIZE * 10) == 0 or done == total:
                print(f'  GPU translated {done}/{total} segments...')

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    return results


def reconstruct_line(seg, translated_pool, idx_holder):
    if seg['type'] == 'blank':
        return seg['line']

    if seg['type'] == 'table_row':
        translated_cells = []
        for cell in seg['cells']:
            if cell['type'] == 'cell':
                tr = translated_pool[idx_holder[0]]
                idx_holder[0] += 1
                if tr is None:
                    tr = cell['content']
                tr = restore_special(tr, cell['protected'])
                translated_cells.append(tr)
            else:
                translated_cells.append(cell['original'])
        return ' ' * seg['indent'] + '|'.join(translated_cells)

    if seg['type'] == 'text':
        tr = translated_pool[idx_holder[0]]
        idx_holder[0] += 1
        if tr is None:
            tr = seg['content']
        tr = restore_special(tr, seg['protected'])
        return ' ' * seg['indent'] + seg['prefix'] + tr

    return ''

print('Translation functions defined.')

Translation functions defined.


In [8]:
#@title 6. 执行翻译
import os

base_name = os.path.splitext(INPUT_FILE)[0]
OUTPUT_FILE = f'{base_name}_cn.md'

print(f'Reading {INPUT_FILE}...')
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    content = f.read()
lines = content.split('\n')
lines = [l + '\n' for l in lines[:-1]] + [lines[-1]] if lines else []

print(f'Total lines: {len(lines)}')

blocks = parse_blocks(lines)
print(f'Total blocks: {len(blocks)}')

text_block_indices = [i for i, (btype, _) in enumerate(blocks) if btype == 'text']
print(f'Text blocks to translate: {len(text_block_indices)}')

print('Phase 1: Extracting segments...')
all_segments = []
block_seg_ranges = {}

for bi in text_block_indices:
    _, bcontent = blocks[bi]
    segs = extract_segments_from_block(bcontent)
    start = len(all_segments)
    all_segments.extend(segs)
    block_seg_ranges[bi] = (start, len(segs))

print('Phase 2: Collecting translatable texts...')
translatable_texts = collect_translatable_texts(all_segments)
non_none = sum(1 for t in translatable_texts if t is not None)
print(f'Total translatable segments: {non_none}')

print('Phase 3: GPU batch translation...')
t0 = time.time()
translated_pool = batch_translate_gpu(translatable_texts)
elapsed = time.time() - t0
print(f'GPU translation done in {elapsed:.1f}s ({non_none/elapsed:.1f} seg/s)')

print('Phase 4: Reconstructing document...')
idx_holder = [0]
output_parts = []

for bi, (btype, bcontent) in enumerate(blocks):
    if btype != 'text':
        output_parts.append(bcontent if bcontent.endswith('\n') or not bcontent else bcontent)
        continue

    start, count = block_seg_ranges[bi]
    result_lines = []
    for seg in all_segments[start:start + count]:
        line = reconstruct_line(seg, translated_pool, idx_holder)
        result_lines.append(line)
    output_parts.append('\n'.join(result_lines))

print(f'Writing {OUTPUT_FILE}...')
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    f.write(''.join(output_parts))

print(f'Translation complete! {len(text_block_indices)} blocks, {non_none} segments in {elapsed:.1f}s')

Reading input (1).md...
Total lines: 17989
Total blocks: 10652
Text blocks to translate: 4870
Phase 1: Extracting segments...
Phase 2: Collecting translatable texts...
Total translatable segments: 12169
Phase 3: GPU batch translation...
  GPU translated 320/12169 segments...
  GPU translated 640/12169 segments...
  GPU translated 960/12169 segments...
  GPU translated 1280/12169 segments...
  GPU translated 1600/12169 segments...
  GPU translated 1920/12169 segments...
  GPU translated 2240/12169 segments...
  GPU translated 2560/12169 segments...
  GPU translated 2880/12169 segments...
  GPU translated 3200/12169 segments...
  GPU translated 3520/12169 segments...
  GPU translated 3840/12169 segments...
  GPU translated 4160/12169 segments...
  GPU translated 4480/12169 segments...
  GPU translated 4800/12169 segments...
  GPU translated 5120/12169 segments...
  GPU translated 5440/12169 segments...
  GPU translated 5760/12169 segments...
  GPU translated 6080/12169 segments...
  GPU 

In [9]:
#@title 7. 下载翻译结果
from google.colab import files

files.download(OUTPUT_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 说明

- **模型**: Helsinki-NLP/opus-mt-en-zh（英→中翻译模型，约300MB）
- **GPU**: Colab T4 GPU，约15GB显存，足够运行此模型
- **批量大小**: 32条/批，自动并行推理
- **保护机制**: 代码块、行内代码、图片引用、LaTeX公式、URL等特殊元素不会被翻译
- **输出格式**: 保持原始Markdown格式，仅翻译文本内容